# 📊 Data Onboarding — `data/real` & `data/ticks`

**Start here.** This notebook teaches the two tick feeds this project runs on, how to load
them, and the gotchas — then points you to the working notebooks. Read top-to-bottom and
run each cell (**Kernel → Restart & Run All**).

| feed | what it is | span | depth ladder? | extra |
|---|---|---|---|---|
| `data/real` | competition **depth dump** (historical) | May 11 – Jun 10 | ✅ full L2 | — |
| `data/ticks` | **live broker logger** (`scripts/log_ticks.py`) | Jun 16 → now | ❌ | `volume` |

`DEFAULT_DATA = [data/real, data/ticks]` stitches them into one continuous feed for
backtests. Everything here is a **thin wrapper over `mt5_trader.*`** — we never re-parse
parquet by hand in research (same loaders run in the backtester and live → parity).

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone

# Run from the repo root no matter where the kernel started (root, or notebooks/).
for _ in range(4):
    if Path("pyproject.toml").exists() and Path("config").is_dir():
        break
    os.chdir("..")
print("cwd =", Path.cwd())

import polars as pl
import matplotlib.pyplot as plt

from mt5_trader.data.ingest import load_ticks
from mt5_trader.data.bars import time_bars
from mt5_trader.features.microstructure import with_micro_features
from mt5_trader.pipeline.data import DEFAULT_DATA, build_bars

SYMBOL = "XAUUSD"
print("DEFAULT_DATA =", [str(d) for d in DEFAULT_DATA])

## 0. Data-present guard

These feeds are **not** committed (big, gitignored). If a cell below complains, get the
data first:
- live ticks → `bash scripts/fetch_logs.sh ticks` (mirrors the VPS tick archive)
- `data/real` (comp dump) is provided separately.

In [ ]:
need = [Path("data/real") / SYMBOL, Path("data/ticks") / SYMBOL]
missing = [str(p) for p in need if not p.exists()]
if missing:
    raise SystemExit(
        f"MISSING {missing}\n"
        "  live ticks:  bash scripts/fetch_logs.sh ticks\n"
        "  data/real :  comp dump, provided separately")
for p in need:
    print(f"OK  {p}  ({len(list(p.glob('*.parquet')))} parquet files)")

## 1. How to load — `load_ticks`

`load_ticks(dirs, symbol, start=, end=)` is the **only** way you should read ticks. Pass a
list of dirs to merge feeds; pass `start`/`end` to slice (the full `data/real` is ~86M
rows — **never load it whole**). Here we grab a couple of hours of `real` and one day of
the live feed.

In [ ]:
real = load_ticks(["data/real"], SYMBOL,
                  start=datetime(2026, 5, 12, 12, tzinfo=timezone.utc),
                  end=datetime(2026, 5, 12, 14, tzinfo=timezone.utc))
live = load_ticks(["data/ticks"], SYMBOL,
                  start=datetime(2026, 6, 16, tzinfo=timezone.utc),
                  end=datetime(2026, 6, 17, tzinfo=timezone.utc))
print("real  (2h):", real.shape)
print("live (1day):", live.shape)
real.head(3)

## 2. Schema tour — and the `CORE_COLUMNS` lesson

`load_ticks` projects **every** feed to a shared schema before merging:

```
CORE_COLUMNS = [ts, symbol, bid, ask, bid_sz, ask_sz]
```

So the loaded view **drops** the live feed's `volume` **and** the real feed's depth ladder.
That's deliberate (one schema to backtest on). To inspect depth or volume you read the raw
files (Section 6). Columns you get from `load_ticks`:

| col | meaning |
|---|---|
| `ts` | UTC timestamp (always UTC here) |
| `bid` / `ask` | best bid / ask price |
| `bid_sz` / `ask_sz` | size at best bid / ask |

In [ ]:
print("loaded (core) schema:")
print(real.schema)
print("\nraw data/real file also has: bidprices[], bidsizes[], askprices[], asksizes[]  (L2 ladder)")
print("raw data/ticks file also has: volume")

## 3. Coverage & gaps

Know your gaps before you backtest. Note the **Jun 11–15 hole** (between the dump and the
live capture) and absent weekends. The strip plot marks every date each feed has.

In [ ]:
def feed_dates(src):
    out = []
    for f in (Path(src) / SYMBOL).glob("*.parquet"):
        s = f.stem
        if s.startswith(SYMBOL):                 # data/real: XAUUSD_2026_05_11
            y, m, d = s.split("_")[-3:]
            out.append(datetime(int(y), int(m), int(d)).date())
        else:                                    # data/ticks: 2026-06-16_<ms>
            out.append(datetime.fromisoformat(s.split("_")[0]).date())
    return sorted(set(out))

real_d, live_d = feed_dates("data/real"), feed_dates("data/ticks")
print(f"data/real : {real_d[0]} .. {real_d[-1]}  ({len(real_d)} days)")
print(f"data/ticks: {live_d[0]} .. {live_d[-1]}  ({len(live_d)} days)")

fig, ax = plt.subplots(figsize=(11, 1.8))
ax.scatter(real_d, [1] * len(real_d), marker="|", s=400, label="data/real")
ax.scatter(live_d, [0] * len(live_d), marker="|", s=400, label="data/ticks")
ax.set_yticks([0, 1]); ax.set_yticklabels(["ticks", "real"])
ax.set_title("date coverage per feed (mind the Jun 11–15 gap)"); ax.legend(loc="center left")
plt.show()

## 4. Spread reality — the number that kills strategies

Cost is the first thing to check. Gold's real-feed spread is **~0.2bp** — but it gaps wide
at session opens, and `data/processed` (Dukascopy) is ~6× wider (wrong feed for cost work).
One full `data/real` day, spread in bp of mid:

In [ ]:
day = load_ticks(["data/real"], SYMBOL,
                 start=datetime(2026, 5, 12, tzinfo=timezone.utc),
                 end=datetime(2026, 5, 13, tzinfo=timezone.utc))
day = day.with_columns(
    spread_bp=((pl.col("ask") - pl.col("bid")) / ((pl.col("ask") + pl.col("bid")) / 2) * 1e4),
    hour=pl.col("ts").dt.hour())
print(day["spread_bp"].describe())

intraday = day.group_by("hour").agg(median_bp=pl.col("spread_bp").median()).sort("hour")
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(intraday["hour"], intraday["median_bp"])
ax.set_xlabel("UTC hour"); ax.set_ylabel("median spread (bp)")
ax.set_title("intraday spread shape — wide at the open"); plt.show()

## 5. Ticks → bars

Strategies see **bars**, not ticks. `time_bars(ticks, every)` aggregates a loaded slice
(OHLC from the mid, plus optional extras like `ofi`). For a full backtest the pipeline's
`build_bars(symbol, every, DEFAULT_DATA)` ingests *all* ticks — heavy, don't call it here;
`time_bars` on a slice is the interactive tool.

In [ ]:
bars = time_bars(day, "5m")
print(bars.select("ts", "open", "high", "low", "close").head(3))

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(bars["ts"], bars["close"])
ax.set_title(f"{SYMBOL} 5m close — one day from data/real"); plt.show()

## 6. Depth ladder & microstructure (real feed only)

The raw `data/real` files keep the full L2 ladder. Read one file directly (small slice) to
see it; `with_micro_features` derives `l1_imbalance` (from best sizes) and `depth_imbalance`
/ `ofi` (need the ladder). **The live feed has no depth — microstructure signals built here
can't deploy live.**

In [ ]:
raw_file = sorted((Path("data/real") / SYMBOL).glob("*.parquet"))[0]
raw = pl.read_parquet(raw_file, n_rows=5000)
print("raw columns:", raw.columns)
row = raw.row(0, named=True)
print("\ntop of book + ladder (first row):")
print("  best bid/ask:", row["bid"], "/", row["ask"])
print("  bid ladder px:", row["bidprices"][:4], "  sizes:", row["bidsizes"][:4])

micro = with_micro_features(raw)
print("\nmicro features:", [c for c in micro.columns if c not in raw.columns])
print(micro.select("ts", "l1_imbalance", "ofi").head(3))

## 7. Where to go next

You now know the feeds. Graduate from here:

- **`data_analysis.ipynb`** — the reusable working template (coverage / spread / depth / backtest).
- **`signal_research.ipynb`** — WORKFLOW Stage 1–2 (IC, decay, by-regime for a signal).
- **`backtesting.ipynb`** — WORKFLOW Stage 3 (equity, cost stress, sweeps, the gate verdict).
- **`docs/WORKFLOW.md`** — the staged research process.
- **`scripts/run_backtest.py`** — `--strategy X --symbol XAUUSD` on `DEFAULT_DATA`.

Discipline: **graduate a validated idea to a registered strategy + the pipeline — don't
leave it living in a cell.** Notebook outputs are stripped on commit by nbstripout.